### ЗАДАЧА: Панель обработки chargeback-кейсов по подпискам по паттерну `MVC`

Финансовая operations-команда разбирает chargeback/dispute-кейсы по подпискам.
Для каждого кейса нужно хранить клиента, подписку, сумму спора, причину,
статус, аналитика и итоговый результат.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила,
- `View` отвечает только за отображение,
- `Controller` принимает действия оператора и связывает слои.

НЕОБХОДИМО:

1. Реализовать сущность `DisputeCase`.
2. Реализовать `DisputeModel`, который умеет:
   - добавлять кейс,
   - назначать аналитика,
   - переводить кейс в `investigating`,
   - помечать evidence как готовое,
   - отправлять кейс в банк,
   - завершать кейс как `won`,
   - завершать кейс как `lost`,
   - возвращать список кейсов для отображения.
3. Бизнес-правила модели:
   - кейс нельзя перевести в `investigating` без аналитика,
   - кейс нельзя отправить в банк, если evidence не подготовлено,
   - кейс можно отправить в банк только из статуса `investigating`,
   - кейс можно завершить как `won` или `lost` только из статуса `submitted`,
   - финальный кейс (`won`, `lost`) нельзя менять дальше.
4. Реализовать `DisputeView` для списка кейсов, успешных сообщений и ошибок.
5. Реализовать `DisputeController`, который вызывает методы модели и передает результат во view.
6. Загрузить начальные кейсы из `initial_cases` в модель.
7. Обработать все действия из `actions` через `controller`.
8. В конце вывести финальное состояние панели dispute-кейсов.


In [10]:
from dataclasses import dataclass


initial_cases = [
    ("DP-700", "sub_1001", "Alice", 19.99, "fraud_claim"),
    ("DP-701", "sub_1002", "Bob", 49.00, "service_not_received"),
]

actions = [
    ("show",),
    ("assign", "DP-700", "Olga"),
    ("investigate", "DP-700"),
    ("submit", "DP-700"),
    ("evidence", "DP-700"),
    ("submit", "DP-700"),
    ("won", "DP-700"),
    ("create", "DP-702", "sub_1003", "Dina", 99.00, "duplicate_charge"),
    ("assign", "DP-702", "Max"),
    ("investigate", "DP-702"),
    ("evidence", "DP-702"),
    ("submit", "DP-702"),
    ("lost", "DP-702"),
    ("show",),
]


@dataclass
class DisputeCase:
    case_id: str
    subscription_id: str
    customer: str
    amount: float
    reason: str
    status: str = "new"
    analyst: str | None = None
    evidence_ready: bool = False
    result: str | None = None


class DisputeModel:
    def __init__(self):
        # TODO 1: создайте self.cases как пустой словарь
        self.cases = {}

    def add_case(self, case_id: str, subscription_id: str, customer: str, amount: float, reason: str) -> None:
        # TODO 2: если case_id уже существует, вызовите ValueError('Case already exists')
        if case_id in self.cases:
            raise ValueError('Case already exists')
        # TODO 3: создайте DisputeCase и сохраните в self.cases
        self.cases[case_id] = DisputeCase(case_id, subscription_id, customer, amount, reason)

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        # TODO 4: если кейс не найден, вызовите ValueError('Case not found')
        if case_id not in self.cases:
            raise ValueError('Case not found')
        # TODO 5: назначьте analyst
        self.cases[case_id].analyst = analyst


    def start_investigation(self, case_id: str) -> None:
        case = self.cases.get(case_id)
        # TODO 6: если кейс не найден, вызовите ValueError('Case not found')
        if case is None:
            raise ValueError('Case not found')
        # TODO 7: если analyst не назначен, вызовите ValueError('Analyst is required')
        if case.analyst is None:
            raise ValueError('Analyst is required')
        # TODO 8: установите status = 'investigating'
        case.status = 'investigating'

    def prepare_evidence(self, case_id: str) -> None:
        case = self.cases.get(case_id)
        # TODO 9: если кейс не найден, вызовите ValueError('Case not found')
        if case is None:
            raise ValueError('Case not found')
        # TODO 10: если status != 'investigating', вызовите ValueError('Case is not investigating')
        if case.status != 'investigating':
            raise ValueError('Case is not investigating')
        # TODO 11: установите evidence_ready = True
        case.evidence_ready = True

    def submit_to_bank(self, case_id: str) -> None:
        case = self.cases.get(case_id)
        # TODO 12: если кейс не найден, вызовите ValueError('Case not found')
        if case is None:
            raise ValueError('Case not found')
        # TODO 13: если status != 'investigating', вызовите ValueError('Case is not investigating')
        if case.status != 'investigating':
            raise ValueError('Case is not investigating')
        # TODO 14: если evidence_ready == False, вызовите ValueError('Evidence is required')
        if case.evidence_ready == False:
            raise ValueError('Evidence is required')
        # TODO 15: установите status = 'submitted'
        case.status = 'submitted'

    def mark_won(self, case_id: str) -> None:
        case = self.cases.get(case_id)
        # TODO 16: если кейс не найден, вызовите ValueError('Case not found')
        if case is None:
            raise ValueError('Case not found')
        # TODO 17: если status != 'submitted', вызовите ValueError('Case is not submitted')
        if case.status != 'submitted':
            raise ValueError('Case is not submitted')
        # TODO 18: установите status = 'won' и result = 'won'
        case.status = 'won'
        case.result = 'won'

    def mark_lost(self, case_id: str) -> None:
        case = self.cases.get(case_id)
        # TODO 19: если кейс не найден, вызовите ValueError('Case not found')
        if case is None:
            raise ValueError('Case not found')
        # TODO 20: если status != 'submitted', вызовите ValueError('Case is not submitted')
        if case.status != 'submitted':
            raise ValueError('Case is not submitted')
        # TODO 21: установите status = 'lost' и result = 'lost'
        case.status = 'lost'
        case.result = 'lost'

    def list_cases(self) -> list[str]:
        # TODO 22: верните список строк формата:
        #   'case_id | subscription_id | amount | status | analyst | evidence_ready | result | reason'
        result = []
        for case_id, case in self.cases.items():
            row = f"{case_id} | {case.subscription_id} | {case.amount} | {case.status} | {case.analyst} | {case.evidence_ready} | {case.result} | {case.reason}"
            result.append(row)
        return result


class DisputeView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        # TODO 23: если rows пустой, выведите 'Кейсов нет'
        if not rows:
            print('Кейсов нет')
        # TODO 24: иначе выведите заголовок 'Dispute cases:' и затем все строки
        else:
            print('Dispute cases:')
            for row in rows:
                print(row)


    @staticmethod
    def render_success(message: str) -> None:
        # TODO 25: выведите сообщение успеха
        print(f"Успех: {message}")

    @staticmethod
    def render_error(message: str) -> None:
        # TODO 26: выведите сообщение ошибки
        print(f"Ошибка: {message}")


class DisputeController:
    def __init__(self, model: DisputeModel, view: DisputeView):
        # TODO 27: сохраните model и view
        self.model = model
        self.view = view

    def create_case(self, case_id: str, subscription_id: str, customer: str, amount: float, reason: str) -> None:
        # TODO 28: через try/except вызовите model.add_case(...)
        # TODO 29: покажите success или error
        try:
            self.model.add_case(case_id, subscription_id, customer, amount, reason)
            self.view.render_success(f"Кейс {case_id} создан")
        except ValueError as e:
            self.view.render_error(str(e))

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        # TODO 30: через try/except вызовите model.assign_analyst(...)
        # TODO 31: покажите success или error
        try:
            self.model.assign_analyst(case_id, analyst)
            self.view.render_success(f"Аналитик {analyst} назначен на кейс {case_id}")
        except ValueError as e:
            self.view.render_error(str(e))

    def start_investigation(self, case_id: str) -> None:
        # TODO 32: через try/except вызовите model.start_investigation(...)
        # TODO 33: покажите success или error
        try:
            self.model.start_investigation(case_id)
            self.view.render_success(f"Кейс {case_id} переведен")
        except ValueError as e:
            self.view.render_error(str(e))

    def prepare_evidence(self, case_id: str) -> None:
        # TODO 34: через try/except вызовите model.prepare_evidence(...)
        # TODO 35: покажите success или error
        try:
            self.model.prepare_evidence(case_id)
            self.view.render_success(f"Кейс {case_id} готов")
        except ValueError as e:
            self.view.render_error(str(e))

    def submit_to_bank(self, case_id: str) -> None:
        # TODO 36: через try/except вызовите model.submit_to_bank(...)
        # TODO 37: покажите success или error
        try:
            self.model.submit_to_bank(case_id)
            self.view.render_success(f"Кейс {case_id} отправлен в банк")
        except ValueError as e:
            self.view.render_error(str(e))

    def mark_won(self, case_id: str) -> None:
        # TODO 38: через try/except вызовите model.mark_won(...)
        # TODO 39: покажите success или error
        try:
            self.model.mark_won(case_id)
            self.view.render_success(f"Завершить {case_id} кейс как 'won'")
        except ValueError as e:
            self.view.render_error(str(e))

    def mark_lost(self, case_id: str) -> None:
        # TODO 40: через try/except вызовите model.mark_lost(...)
        # TODO 41: покажите success или error
        try:
            self.model.mark_lost(case_id)
            self.view.render_success(f"Завершить {case_id} кейс как 'lost'")
        except ValueError as e:
            self.view.render_error(str(e))

    def show_cases(self) -> None:
        # TODO 42: получите данные из model.list_cases() и передайте их во view.render_cases(...)
        rows = self.model.list_cases()
        self.view.render_cases(rows)


model = DisputeModel()
view = DisputeView()
controller = DisputeController(model, view)

# TODO 43: загрузите начальные кейсы из initial_cases в model
# Подсказка: пройдите по initial_cases циклом for
# и вызовите model.add_case(case_id, subscription_id, customer, amount, reason)
for case_data in initial_cases:
    case_id, subscription_id, customer, amount, reason = case_data
    model.add_case(case_id, subscription_id, customer, amount, reason)
    

# TODO 44: обработайте все действия из actions
for action in actions:
    action_type = action[0]
    # TODO 45: если action[0] == 'show', вызовите controller.show_cases()
    if action[0] == 'show':
        controller.show_cases()
    # TODO 46: если action[0] == 'create', распакуйте case_id, subscription_id, customer, amount, reason
    # и вызовите controller.create_case(...)
    if action[0] == 'create':
        _, case_id, subscription_id, customer, amount, reason = action
        controller.create_case(case_id, subscription_id, customer, amount, reason)
    # TODO 47: если action[0] == 'assign', распакуйте case_id и analyst
    # и вызовите controller.assign_analyst(...)
    if action[0] == 'assign':
        _, case_id, analyst = action
        controller.assign_analyst(case_id, analyst)
    # TODO 48: если action[0] == 'investigate', распакуйте case_id
    # и вызовите controller.start_investigation(...)
    if action[0] == 'investigate':
        _, case_id = action
        print(action)
        controller.start_investigation(case_id)
    # TODO 49: если action[0] == 'evidence', распакуйте case_id
    # и вызовите controller.prepare_evidence(...)
    if action[0] == 'evidence':
        _, case_id = action
        controller.prepare_evidence(case_id)
    # TODO 50: если action[0] == 'submit', распакуйте case_id
    # и вызовите controller.submit_to_bank(...)
    if action[0] == 'submit':
        _, case_id = action
        controller.submit_to_bank(case_id)
    # TODO 51: если action[0] == 'won', распакуйте case_id
    # и вызовите controller.mark_won(...)
    if action[0] == 'won':
        _, case_id = action
        controller.mark_won(case_id)
    # TODO 52: если action[0] == 'lost', распакуйте case_id
    # и вызовите controller.mark_lost(...)
    if action[0] == 'lost':
        _, case_id = action
        controller.mark_lost(case_id)

print()
print("Финальное состояние dispute-кейсов:")
# TODO 53: покажите финальное состояние панели через controller.show_cases()
controller.show_cases()


Dispute cases:
DP-700 | sub_1001 | 19.99 | new | None | False | None | fraud_claim
DP-701 | sub_1002 | 49.0 | new | None | False | None | service_not_received
Успех: Аналитик Olga назначен на кейс DP-700
('investigate', 'DP-700')
Успех: Кейс DP-700 переведен
Ошибка: Evidence is required
Успех: Кейс DP-700 готов
Успех: Кейс DP-700 отправлен в банк
Успех: Завершить DP-700 кейс как 'won'
Успех: Кейс DP-702 создан
Успех: Аналитик Max назначен на кейс DP-702
('investigate', 'DP-702')
Успех: Кейс DP-702 переведен
Успех: Кейс DP-702 готов
Успех: Кейс DP-702 отправлен в банк
Успех: Завершить DP-702 кейс как 'lost'
Dispute cases:
DP-700 | sub_1001 | 19.99 | won | Olga | True | won | fraud_claim
DP-701 | sub_1002 | 49.0 | new | None | False | None | service_not_received
DP-702 | sub_1003 | 99.0 | lost | Max | True | lost | duplicate_charge

Финальное состояние dispute-кейсов:
Dispute cases:
DP-700 | sub_1001 | 19.99 | won | Olga | True | won | fraud_claim
DP-701 | sub_1002 | 49.0 | new | None | 